# Travelling map

In [258]:
!pip install folium
!pip install gpxpy

In [1]:
import folium 
import gpxpy
import os
import json
from branca.element import MacroElement, Template

In [2]:
m = folium.Map(location=[28, 60], zoom_start=2)

In [3]:
class ClickableGPXLine(MacroElement):
    """
    Adds Leaflet JS behavior:
    - click line once: highlight it and open popup
    - click same line again: restore original color and close popup
    - click another line: restore previous line, highlight new line
    """

    _template = Template("""
    {% macro script(this, kwargs) %}
    (function() {
        var map = {{ this._parent.get_name() }};
        var layer = {{ this.layer_name }};

        var normalStyle = {{ this.normal_style }};
        var clickedStyle = {{ this.clicked_style }};

        if (!map._activeGPXLineState) {
            map._activeGPXLineState = {
                activeLayer: null
            };
        }

        layer._normalGPXStyle = normalStyle;

        layer.on('click', function(e) {
            var state = map._activeGPXLineState;

            // If another line is active, reset it first
            if (state.activeLayer && state.activeLayer !== layer) {
                state.activeLayer.setStyle(state.activeLayer._normalGPXStyle);
                state.activeLayer.closePopup();
            }

            // If this same line is clicked again, toggle it off
            if (state.activeLayer === layer) {
                layer.setStyle(normalStyle);
                layer.closePopup();
                state.activeLayer = null;
            } else {
                layer.setStyle(clickedStyle);
                layer.openPopup(e.latlng);
                state.activeLayer = layer;
            }
        });
    })();
    {% endmacro %}
    """)

    def __init__(self, layer, normal_style, clicked_style):
        super().__init__()
        self.layer_name = layer.get_name()
        self.normal_style = json.dumps(normal_style)
        self.clicked_style = json.dumps(clicked_style)

In [4]:
def overlayGPX(
    gpxData,
    fmap,
    color="blue",
    weight=4,
    popup_text="GPX route",
    clicked_color="red",
    clicked_weight=None,
):
    """
    Overlay a clickable GPX route on a Folium map.

    Parameters
    ----------
    gpxData : str
        Path to GPX file.
    fmap : folium.Map
        Existing Folium map.
    color : str
        Normal line color.
    weight : int or float
        Normal line weight.
    popup_text : str
        Text shown when the line is clicked.
    clicked_color : str
        Color used when the line is selected.
    clicked_weight : int or float, optional
        Weight used when selected. Defaults to original weight.
    """

    with open(gpxData, "r", encoding="utf-8") as gpx_file:
        gpx = gpxpy.parse(gpx_file)

    points = []

    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                points.append((point.latitude, point.longitude))

    if not points:
        raise ValueError(f"No track points found in GPX file: {gpxData}")

    normal_style = {
        "color": color,
        "weight": weight,
        "opacity": 1,
    }

    clicked_style = {
        "color": clicked_color,
        "weight": clicked_weight if clicked_weight is not None else weight,
        "opacity": 1,
    }

    line = folium.PolyLine(
        points,
        color=color,
        weight=weight,
        opacity=1,
    )
    
    popup_html = f"""
    <div style="
        display: inline-block;
        width: max-content;
        min-width: 160px;
        max-width: 600px;
        white-space: nowrap;
    ">
        {popup_text}
    </div>
    """
    
    line.add_child(
        folium.Popup(
            popup_html,
            max_width=650
        )
    )
    
    line.add_to(fmap)

    click_handler = ClickableGPXLine(
        layer=line,
        normal_style=normal_style,
        clicked_style=clicked_style,
    )

    fmap.add_child(click_handler)

    return line

In [5]:
# 1. Russia (1997)
# 2. Great Britain (2012)
# 3. Finland (2014)
# 4. Belgium (2016)
# 5. South Africa (2016)
# 6. Georgia (2019)
# 7. Kyrgyzstan (2022)
# 8. Armenia (2022)
# 9. Serbia (2022)
# 10. Singapore (2022)
# 11. Cyprus (2022)
# 12. Australia (2023)
# 13. Luxembourg (2023)
# 14. France (2023)
# 15. Germany (2023)
# 16. Portugal (2024)
# 17. Turkiye (2025)
# 18. Egypt (2025)
# 19. Botswana (2025)
# 20. Bosnia and Herzegovina (2026)
# 21. Brazil (2026)

In [6]:
# London / studies / MP Education
overlayGPX("flights/2012-11-04-petersburg-stockholm.gpx", m, "grey", 3, "<b>Saint Petersburg — Stockholm</b><br>November 4th, 2012", "darkblue", 5) # SK733
overlayGPX("flights/2012-11-04-stockholm-london.gpx", m, "grey", 3, "<b>Stockholm — London</b><br>November 4th, 2012", "darkblue", 5) # SK1527
overlayGPX("flights/2012-11-10-london-frankfurt.gpx", m, "grey", 3, "<b>London — Frankfurt am Main</b><br>November 10th, 2012", "darkblue", 5) # LH903
overlayGPX("flights/2012-11-10-frankfurt-petersburg.gpx", m, "grey", 3, "<b>Frankfurt am Main — Saint Petersburg</b><br>November 10th, 2012", "darkblue", 5) # LH1438

# Brussels / studies / ITMO International Council (+ getting SA visa)
overlayGPX("flights/2016-11-10-petersburg-moscow.gpx", m, "grey", 3, "<b>Saint Petersburg — Moscow</b><br>November 10th, 2016", "darkblue", 5) # SU35
overlayGPX("flights/2016-11-10-moscow-brussels.gpx", m, "grey", 3, "<b>Moscow — Brussels</b><br>November 10th, 2016", "darkblue", 5) # SU2168
overlayGPX("flights/2016-11-13-brussels-moscow.gpx", m, "grey", 3, "<b>Brussels — Moscow</b><br>November 13th, 2016", "darkblue", 5) # SU2619
overlayGPX("flights/2016-11-14-moscow-petersburg.gpx", m, "grey", 3, "<b>Moscow — Saint Petersburg</b><br>November 14th, 2016", "darkblue", 5) # SU6026

# Johannesburg / studies / BRICS summit
overlayGPX("flights/2016-11-27-petersburg-moscow.gpx", m, "grey", 3, "<b>Saint Petersburg — Moscow</b><br>November 27th, 2016", "darkblue", 5) # S744
overlayGPX("flights/2016-11-27-moscow-doha.gpx", m, "grey", 3, "<b>Moscow — Doha</b><br>November 27th, 2016", "darkblue", 5) # QR230
overlayGPX("flights/2016-11-28-doha-johannesburg.gpx", m, "grey", 3, "<b>Doha — Johannesburg</b><br>November 28th, 2016", "darkblue", 5) # QR1367
overlayGPX("flights/2016-12-03-johannesburg-dubai.gpx", m, "grey", 3, "<b>Johannesburg — Dubai</b><br>December 3rd, 2016", "darkblue", 5) # EK764
overlayGPX("flights/2016-12-04-dubai-petersburg.gpx", m, "grey", 3, "<b>Dubai — Saint Petersburg</b><br>December 4th, 2016", "darkblue", 5) # EK175

# Tomsk / travel / Visiting Lisa
overlayGPX("flights/2019-08-25-petersburg-moscow.gpx", m, "grey", 3, "<b>Saint Petersburg — Moscow</b><br>August 25th, 2019", "darkblue", 5) # U680
overlayGPX("flights/2019-08-25-moscow-tomsk.gpx", m, "grey", 3, "<b>Moscow — Tomsk</b><br>August 25th, 2019", "darkblue", 5) # U611
overlayGPX("flights/2019-08-31-tomsk-moscow.gpx", m, "grey", 3, "<b>Tomsk — Moscow</b><br>August 31th, 2019", "darkblue", 5) # SU1531
overlayGPX("flights/2019-08-31-moscow-petersburg.gpx", m, "grey", 3, "<b>Moscow — Saint Petersburg</b><br>August 31th, 2019", "darkblue", 5) # SU42

# Tbilisi / work / getting US visa
overlayGPX("flights/2019-10-20-petersburg-minsk.gpx", m, "grey", 3, "<b>Saint Petersburg — Minsk</b><br>October 20th, 2019", "darkblue", 5) # B2944
overlayGPX("flights/2019-10-20-minsk-tbilisi.gpx", m, "grey", 3, "<b>Minsk — Tbilisi</b><br>October 20th, 2019", "darkblue", 5) # B2741
overlayGPX("flights/2019-10-26-tbilisi-minsk.gpx", m, "grey", 3, "<b>Tbilisi — Minsk</b><br>October 26th, 2019", "darkblue", 5) # B2736
overlayGPX("flights/2019-10-26-minsk-petersburg.gpx", m, "grey", 3, "<b>Minsk — Saint Petersburg</b><br>October 26th, 2019", "darkblue", 5) # B2941

# Bishkek / travel / Sitting it out
overlayGPX("flights/2022-03-03-petersburg-moscow.gpx", m, "grey", 3, "<b>Saint Petersburg — Moscow</b><br>March 3rd, 2022", "darkblue", 5) # SU31
overlayGPX("flights/2022-03-04-moscow-bishkek.gpx", m, "grey", 3, "<b>Moscow — Bishkek</b><br>March 3rd, 2022", "darkblue", 5) # SU1880
overlayGPX("flights/2022-03-14-bishkek-moscow.gpx", m, "grey", 3, "<b>Bishkek — Moscow</b><br>March 14th, 2022", "darkblue", 5) # YK959

# Yerevan / relocation
overlayGPX("flights/2022-04-05-moscow-yerevan.gpx", m, "grey", 3, "<b>Moscow — Yerevan</b><br>April 5th, 2022", "darkblue", 5) # 3F322

# Serbia / relocation
overlayGPX("flights/2022-06-11-yerevan-frankfurt.gpx", m, "grey", 3, "<b>Yerevan — Frankfurt am Main</b><br>June 11th, 2022", "darkblue", 5) # LH1561
overlayGPX("flights/2022-06-11-frankfurt-belgrade.gpx", m, "grey", 3, "<b>Frankfurt am Main — Belgrade</b><br>June 11th, 2022", "darkblue", 5) # LH1406

# Singapore / work / FSE'22
overlayGPX("flights/2022-11-12-belgrade-doha.gpx", m, "grey", 3, "<b>Belgrade — Doha</b><br>November 12th, 2022", "darkblue", 5) # QR232
overlayGPX("flights/2022-11-12-doha-singapore.gpx", m, "grey", 3, "<b>Doha — Singapore</b><br>November 12th, 2022", "darkblue", 5) # QR942
overlayGPX("flights/2022-11-19-singapore-doha.gpx", m, "grey", 3, "<b>Singapore — Doha</b><br>November 19th, 2022", "darkblue", 5) # QR947
overlayGPX("flights/2022-11-20-doha-belgrade.gpx", m, "grey", 3, "<b>Doha — Belgrade</b><br>November 20th, 2022", "darkblue", 5) # QR231

# Cyprus / work / gathering
overlayGPX("flights/2022-12-11-belgrade-larnaca.gpx", m, "grey", 3, "<b>Belgrade — Larnaca</b><br>December 11th, 2022", "darkblue", 5) # JU880
overlayGPX("flights/2022-12-17-larnaca-belgrade.gpx", m, "grey", 3, "<b>Larnaca — Belgrade</b><br>December 17th, 2022", "darkblue", 5) # JU887
 
# Melbourne / work / ICSE'23
overlayGPX("flights/2023-05-12-belgrade-doha.gpx", m, "grey", 3, "<b>Belgrade — Doha</b><br>May 12th, 2023", "darkblue", 5) # QR232
overlayGPX("flights/2023-05-12-doha-melbourne.gpx", m, "grey", 3, "<b>Doha — Melbourne</b><br>May 12th, 2023", "darkblue", 5) # QR904
overlayGPX("flights/2023-05-21-melbourne-doha.gpx", m, "grey", 3, "<b>Melbourne — Doha</b><br>May 21st, 2023", "darkblue", 5) # QR905
overlayGPX("flights/2023-05-22-doha-belgrade.gpx", m, "grey", 3, "<b>Doha — Belgrade</b><br>May 22nd, 2023", "darkblue", 5) # QR231

# Luxembourg / work / ASE'23
overlayGPX("flights/2023-09-10-belgrade-luxembourg.gpx", m, "grey", 3, "<b>Belgrade — Luxembourg</b><br>September 10th, 2023", "darkblue", 5) # LG8114
overlayGPX("flights/2023-09-16-luxembourg-paris.gpx", m, "grey", 3, "<b>Luxembourg — Paris</b><br>September 16th, 2023", "darkblue", 5) # LG8021
overlayGPX("flights/2023-09-16-paris-belgrade.gpx", m, "grey", 3, "<b>Paris — Belgrade</b><br>September 16th, 2023", "darkblue", 5) # JU315

# Lisbon / work / ICSE'24
overlayGPX("flights/2024-04-14-belgrade-frankfurt.gpx", m, "grey", 3, "<b>Belgrade — Frankfurt am Main</b><br>April 14th, 2024", "darkblue", 5) # LH1407
overlayGPX("flights/2024-04-14-frankfurt-lisbon.gpx", m, "grey", 3, "<b>Frankfurt am Main — Lisbon</b><br>April 14th, 2024", "darkblue", 5) # LH1496
overlayGPX("flights/2024-04-21-lisbon-frankfurt.gpx", m, "grey", 3, "<b>Lisbon — Frankfurt am Main</b><br>April 21st, 2024", "darkblue", 5) # LH1169
overlayGPX("flights/2024-04-21-frankfurt-belgrade.gpx", m, "grey", 3, "<b>Frankfurt am Main — Belgrade</b><br>April 21st, 2024", "darkblue", 5) # LH1410

# Cyprus / work / gathering
overlayGPX("flights/2024-11-17-belgrade-larnaca.gpx", m, "grey", 3, "<b>Belgrade — Larnaca</b><br>November 17th, 2024", "darkblue", 5) # JU480
overlayGPX("flights/2024-11-23-larnaca-belgrade.gpx", m, "grey", 3, "<b>Larnaca — Belgrade</b><br>November 23rd, 2024", "darkblue", 5) # W64030

# Istanbul, Cairo / travel
overlayGPX("flights/2025-01-30-belgrade-istanbul.gpx", m, "grey", 3, "<b>Belgrade — Istanbul</b><br>January 30th, 2025", "darkblue", 5) # TK1082
overlayGPX("flights/2025-02-04-istanbul-cairo.gpx", m, "grey", 3, "<b>Istanbul — Cairo</b><br>February 4th, 2025", "darkblue", 5) # TK694
overlayGPX("flights/2025-02-09-cairo-istanbul.gpx", m, "grey", 3, "<b>Cairo — Istanbul</b><br>February 9th, 2025", "darkblue", 5) # TK695
overlayGPX("flights/2025-02-10-istanbul-belgrade.gpx", m, "grey", 3, "<b>Istanbul — Belgrade</b><br>February 10th, 2025", "darkblue", 5) # TK1081

# Gaborone / work / CompEd'25
overlayGPX("flights/2025-10-21-belgrade-dubai.gpx", m, "grey", 3, "<b>Belgrade — Dubai</b><br>October 21st, 2025", "darkblue", 5) # FZ1746
overlayGPX("flights/2025-10-21-dubai-johannesburg.gpx", m, "grey", 3, "<b>Dubai — Johannesburg</b><br>October 21st, 2025", "darkblue", 5) # EK767
overlayGPX("flights/2025-10-22-johannesburg-gaborone.gpx", m, "grey", 3, "<b>Johannesburg — Gaborone</b><br>October 22nd, 2025", "darkblue", 5) # 4Z174 
overlayGPX("flights/2025-10-26-gaborone-johannesburg.gpx", m, "grey", 3, "<b>Gaborone — Johannesburg</b><br>October 26th, 2025", "darkblue", 5) # 4Z177
overlayGPX("flights/2025-10-26-johannesburg-dubai.gpx", m, "grey", 3, "<b>Johannesburg — Dubai</b><br>October 26th, 2025", "darkblue", 5) # EK764
overlayGPX("flights/2025-10-27-dubai-belgrade.gpx", m, "grey", 3, "<b>Dubai — Belgrade</b><br>October 27th, 2025", "darkblue", 5) # FZ1745

# Sarajevo / travel / Pasha's birthday
overlayGPX("flights/2026-03-13-belgrade-sarajevo.gpx", m, "grey", 3, "<b>Belgrade — Sarajevo</b><br>March 13th, 2026", "darkblue", 5) # JU652
overlayGPX("flights/2026-03-15-sarajevo-belgrade.gpx", m, "grey", 3, "<b>Sarajevo — Belgrade</b><br>March 15th, 2026", "darkblue", 5) # JU653

# Rio de Janeiro / work / ICSE'26
overlayGPX("flights/2026-04-11-belgrade-amsterdam.gpx", m, "grey", 3, "<b>Belgrade — Amsterdam</b><br>April 11th, 2026", "darkblue", 5) # KL1982
overlayGPX("flights/2026-04-11-amsterdam-rio.gpx", m, "grey", 3, "<b>Amsterdam — Rio de Janeiro</b><br>April 11th, 2026", "darkblue", 5) # KL705
overlayGPX("flights/2026-04-19-rio-amsterdam.gpx", m, "grey", 3, "<b>Rio de Janeiro — Amsterdam</b><br>April 19th, 2026", "darkblue", 5) # KL706
overlayGPX("flights/2026-04-20-amsterdam-belgrade.gpx", m, "grey", 3, "<b>Amsterdam — Belgrade</b><br>April 20th, 2026", "darkblue", 5) # KL1985

In [7]:
# Glebychevo / travel
overlayGPX("trains/1999-06-01-petersburg-glebychevo.gpx", m, "grey", 3, "<b>Saint Petersburg — Gebychevo</b> and back<br>June 1999", "darkred", 5)

# Tula / studies
overlayGPX("trains/2013-10-10-petersburg-tula.gpx", m, "grey", 3, "<b>Saint Petersburg — Tula</b> and back<br>October 2013", "darkred", 5)

# Otradnoye / travel
overlayGPX("trains/2014-06-10-petersburg-otradnoye.gpx", m, "grey", 3, "<b>Saint Petersburg — Otradnoye</b> and back<br>June 10–20th, 2014, and several more times", "darkred", 5)

# Lebyazhye / travel
overlayGPX("trains/2015-06-27-petersburg-lebyazhye.gpx", m, "grey", 3, "<b>Saint Petersburg — Lebyazhye</b> and back<br>June 27th, 2015, and several more times", "darkred", 5)

# Moscow / travel
overlayGPX("trains/2016-02-01-petersburg-moscow.gpx", m, "grey", 3, "<b>Saint Petersburg — Moscow</b> and back<br>February 1–4th, 2016, and several more times", "darkred", 5)

# Velikiy Novgorod / travel

overlayGPX("trains/2016-05-01-petersburg-novgorod.gpx", m, "grey", 3, "<b>Saint Petersburg — Veliky Novgorod</b> and back<br>May 1st, 2016", "darkred", 5)

# Belgium / travel
overlayGPX("trains/2016-11-11-brussels-waterloo.gpx", m, "grey", 3, "<b>Brussels — Waterloo</b> and back<br>November 11th, 2016", "darkred", 5)
overlayGPX("trains/2016-11-11-brussels-bruges.gpx", m, "grey", 3, "<b>Brussels — Bruges</b> and back<br>November 11th, 2016", "darkred", 5)

# Ladozhskoye ozero / travel
overlayGPX("trains/2017-01-30-petersburg-ladoga.gpx", m, "grey", 3, "<b>Saint Petersburg — Ladozhskoye ozero</b> and back<br>January 30th, 2017, and several more times", "darkred", 5)

# Nizhy Novgorod / travel
overlayGPX("trains/2018-01-30-petersburg-novgorod.gpx", m, "grey", 3, "<b>Saint Petersburg — Nizhny Novgorod</b> and back<br>January 30th – February 4th, 2018", "darkred", 5)

# Kuznechnoye / travel
overlayGPX("trains/2018-06-10-petersburg-kuznechnoye.gpx", m, "grey", 3, "<b>Saint Petersburg — Kuznechnoye</b> and back<br>June 10th, 2018", "darkred", 5)

# Kaliningrad / travel
overlayGPX("trains/2018-08-16-petersburg-kaliningrad.gpx", m, "grey", 3, "<b>Saint Petersburg — Kaliningrad</b> and back<br>August 15–21st, 2018", "darkred", 5)

# Kaluga / travel
overlayGPX("trains/2019-05-05-moscow-kaluga.gpx", m, "grey", 3, "<b>Moscow — Kaluga</b> and back<br>May 4–5th, 2019", "darkred", 5)

# Kirovsk / travel
overlayGPX("trains/2019-11-24-petersburg-kirovsk.gpx", m, "grey", 3, "<b>Saint Petersburg — Kirovsk</b> and back<br>November 24th, 2019", "darkred", 5)

# Simferopol / travel
overlayGPX("trains/2020-09-20-petersburg-simferopol.gpx", m, "grey", 3, "<b>Saint Petersburg — Simferopol</b> and back<br>September 20–30th, 2020", "darkred", 5)

# Sortavala / travel
overlayGPX("trains/2021-05-08-petersburg-sortavala.gpx", m, "grey", 3, "<b>Saint Petersburg — Sortavala</b> and back<br>May 8–10th, 2021", "darkred", 5)

# Pskov / travel
overlayGPX("trains/2021-06-21-petersburg-pskov.gpx", m, "grey", 3, "<b>Saint Petersburg — Pskov</b> and back<br>June 21–23rd, 2021", "darkred", 5)

# Paris / travel
overlayGPX("trains/2023-09-10-luxembourg-paris.gpx", m, "grey", 3, "<b>Luxembourg — Paris</b> and back<br>September 10–11th, 2023", "darkred", 5)

# Novi Sad / travel
overlayGPX("trains/2023-11-04-belgrade-novisad.gpx", m, "grey", 3, "<b>Belgrade — Novi Sad</b> and back<br>November 4th, 2023, and several more times", "darkred", 5)

# Egypt / travel
overlayGPX("trains/2025-02-06-cairo-luxor.gpx", m, "grey", 3, "<b>Cairo — Luxor</b> and back<br>February 6–8th, 2025", "darkred", 5)
overlayGPX("trains/2025-02-08-cairo-alexandria.gpx", m, "grey", 3, "<b>Cairo — Alexandria</b> and back<br>February 8–9th, 2025", "darkred", 5)

In [8]:
# Izhora / travel
overlayGPX("ships/2002-05-01-vaya-kommunar.gpx", m, "grey", 3, "<b>Vaya — Kommunar</b><br>May 2022", "darkgreen", 5)

In [9]:
def dynamic_width_popup(html, min_width=80, max_width=600):
    popup_html = f"""
    <div style="
        display: inline-block;
        width: max-content;
        min-width: {min_width}px;
        max-width: {max_width}px;
        white-space: nowrap;
    ">
        {html}
    </div>
    """

    return folium.Popup(
        popup_html,
        max_width=max_width + 50
    )

color_home = "red"
icon_home = "fa-home"
z_index_home = 2000

color_living = "orange"
icon_living = "fa-home"
z_index_living = 1000

color_work = "darkpurple"
icon_work = "briefcase"
z_index_work = 500

color_studies = "blue"
icon_studies = "university"
z_index_studies = 300

color_travel = "green"
icon_travel = "suitcase-rolling"
z_index_travel = 200

color_transit = "lightgray"
icon_transit = "plane"
z_index_transit = 100

color_train = "lightgray"
icon_train = "subway"
z_index_train = 100

In [10]:
folium.Marker(
    [59.938948, 30.315950], 
    popup=dynamic_width_popup("<b>Saint Petersburg</b><br>April 1997"),  
    icon=folium.Icon(color=color_home, icon=icon_home, prefix="fa"), z_index_offset=z_index_home
).add_to(m)

folium.Marker(
    [59.717387, 30.397206], 
    popup=dynamic_width_popup("<b>Pushkin</b><br>June 2004"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.682742, 30.449041], 
    popup=dynamic_width_popup("<b>Pavlovsk</b><br>December 2007"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [51.500738, -0.124571], 
    popup=dynamic_width_popup("<b>London</b><br>November 2012"), 
    icon=folium.Icon(color=color_studies, icon=icon_studies, prefix="fa"), z_index_offset=z_index_studies
).add_to(m)

folium.Marker(
    [51.145870, 0.874715], 
    popup=dynamic_width_popup("<b>Ashford</b><br>November 2012"), 
    icon=folium.Icon(color=color_studies, icon=icon_studies, prefix="fa"), z_index_offset=z_index_studies
).add_to(m)

folium.Marker(
    [54.19911, 37.57748], 
    popup=dynamic_width_popup("<b>Tula</b><br>October 2013"), 
    icon=folium.Icon(color=color_train, icon=icon_train, prefix="fa"), z_index_offset=z_index_train
).add_to(m)

folium.Marker(
    [54.070540, 37.526802], 
    popup=dynamic_width_popup("<b>Yasnaya Polyana</b><br>October 2013"), 
    icon=folium.Icon(color=color_studies, icon=icon_studies, prefix="fa"), z_index_offset=z_index_studies
).add_to(m)

folium.Marker(
    [53.369554, 36.629296], 
    popup=dynamic_width_popup("<b>Spasskoye-Lutovinovo</b><br>October 2013"), 
    icon=folium.Icon(color=color_studies, icon=icon_studies, prefix="fa"), z_index_offset=z_index_studies
).add_to(m)

folium.Marker(
    [59.772583, 30.791796], 
    popup=dynamic_width_popup("<b>Otradnoye</b><br>June 2014"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [61.868079, 28.885904], 
    popup=dynamic_width_popup("<b>Savonlinna</b><br>July 2014"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.867882, 26.703927], 
    popup=dynamic_width_popup("<b>Kouvola</b><br>August 2014"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.742486, 30.594710], 
    popup=dynamic_width_popup("<b>Kolpino</b><br>February 2015"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [55.753403, 37.620834], 
    popup=dynamic_width_popup("<b>Moscow</b><br>February 2015"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.095787, 29.954779], 
    popup=dynamic_width_popup("<b>Sestroretsk</b><br>March 2015"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.561738, 30.115215], 
    popup=dynamic_width_popup("<b>Gatchina</b><br>September 2015"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.192213, 29.701862], 
    popup=dynamic_width_popup("<b>Zelenogorsk</b><br>April 2016"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [58.521272, 31.276110], 
    popup=dynamic_width_popup("<b>Veliky Novgorod</b><br>May 2016"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [50.846792, 4.352176], 
    popup=dynamic_width_popup("<b>Brussels</b><br>November 2016"), 
    icon=folium.Icon(color=color_studies, icon=icon_studies, prefix="fa"), z_index_offset=z_index_studies
).add_to(m)

folium.Marker(
    [51.208326, 3.226681], 
    popup=dynamic_width_popup("<b>Bruges</b><br>November 2016"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [50.717609, 4.398007], 
    popup=dynamic_width_popup("<b>Waterloo</b><br>November 2016"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [-26.192572, 28.036835], 
    popup=dynamic_width_popup("<b>Johannesburg</b><br>November 2016"), 
    icon=folium.Icon(color=color_studies, icon=icon_studies, prefix="fa"), z_index_offset=z_index_studies
).add_to(m)

folium.Marker(
    [60.167582, 24.942142], 
    popup=dynamic_width_popup("<b>Helsinki</b><br>May 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.393130, 25.665340], 
    popup=dynamic_width_popup("<b>Porvoo</b><br>May 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.991425, 29.778873], 
    popup=dynamic_width_popup("<b>Kronstadt</b><br>June 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [56.310660, 38.130471], 
    popup=dynamic_width_popup("<b>Sergiev Posad</b><br>July 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [56.736588, 38.852133], 
    popup=dynamic_width_popup("<b>Pereslavl-Zalessky</b><br>July 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [57.184736, 39.416595], 
    popup=dynamic_width_popup("<b>Rostov</b><br>July 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [57.621452, 39.888206], 
    popup=dynamic_width_popup("<b>Yaroslavl</b><br>July 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [57.767677, 40.926229], 
    popup=dynamic_width_popup("<b>Kostroma</b><br>July 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [57.001257, 40.973717], 
    popup=dynamic_width_popup("<b>Ivanovo</b><br>July 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [56.138433, 40.399936], 
    popup=dynamic_width_popup("<b>Vladimir</b><br>July 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [56.421229, 40.447081], 
    popup=dynamic_width_popup("<b>Suzdal</b><br>July 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [54.706214, 20.512923], 
    popup=dynamic_width_popup("<b>Kaliningrad</b><br>August 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [54.640040, 21.806177], 
    popup=dynamic_width_popup("<b>Chernyakhovsk</b><br>August 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [54.589787, 22.202236], 
    popup=dynamic_width_popup("<b>Gusev</b><br>August 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [54.644043, 19.893790], 
    popup=dynamic_width_popup("<b>Baltiysk</b><br>August 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [54.868306, 19.938658], 
    popup=dynamic_width_popup("<b>Yantarny</b><br>August 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [54.959545, 20.479084], 
    popup=dynamic_width_popup("<b>Zelenogradsk</b><br>August 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [54.514392, 36.256562], 
    popup=dynamic_width_popup("<b>Kaluga</b><br>May 2019"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [57.821731, 28.329065], 
    popup=dynamic_width_popup("<b>Pskov</b><br>June 2021"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [57.022699, 28.914927], 
    popup=dynamic_width_popup("<b>Pushkinskiye Gory</b><br>June 2021"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.881924, 30.983610], 
    popup=dynamic_width_popup("<b>Kirovsk</b><br>November 2019"), 
    icon=folium.Icon(color=color_living, icon=icon_living, prefix="fa"), z_index_offset=z_index_living
).add_to(m)

folium.Marker(
    [59.944291, 31.033289], 
    popup=dynamic_width_popup("<b>Shlisselburg</b><br>December 2019"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.712926, 28.733257], 
    popup=dynamic_width_popup("<b>Vyborg</b><br>October 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [61.699749, 30.692326], 
    popup=dynamic_width_popup("<b>Sortavala</b><br>May 2021"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [61.388734, 30.944909], 
    popup=dynamic_width_popup("<b>Valaam</b><br>May 2021"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [58.731808, 29.848915], 
    popup=dynamic_width_popup("<b>Luga</b><br>July 2019"), 
    icon=folium.Icon(color=color_studies, icon=icon_studies, prefix="fa"), z_index_offset=z_index_studies
).add_to(m)

folium.Marker(
    [41.716969, 44.787393], 
    popup=dynamic_width_popup("<b>Tbilisi</b><br>October 2019"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [56.475355, 84.950201], 
    popup=dynamic_width_popup("<b>Tomsk</b><br>August 2019"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [56.326158, 44.003545], 
    popup=dynamic_width_popup("<b>Nizhny Novgorod</b><br>February 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.878285, 29.892599], 
    popup=dynamic_width_popup("<b>Petergof</b><br>October 2012"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.452910, 28.721317], 
    popup=dynamic_width_popup("<b>Glebychevo</b><br>June 1999"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.954031, 29.098551], 
    popup=dynamic_width_popup("<b>Kandikyulya</b><br>June 2001"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.601187, 30.160734], 
    popup=dynamic_width_popup("<b>Vaya</b><br>May 2002"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.628242, 30.414084], 
    popup=dynamic_width_popup("<b>Kommunar</b><br>May 2002"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.350843, 30.070585], 
    popup=dynamic_width_popup("<b>Siversky</b><br>April 2004"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.410492, 30.340363], 
    popup=dynamic_width_popup("<b>Vyritsa</b><br>April 2004"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.494574, 30.268424], 
    popup=dynamic_width_popup("<b>Orekhovo</b><br>June 2006"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.382123, 30.355684], 
    popup=dynamic_width_popup("<b>Vaskelovo</b><br>July 2008"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.682680, 30.064464], 
    popup=dynamic_width_popup("<b>Ovragi</b><br>May 2009"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.359286, 28.990139], 
    popup=dynamic_width_popup("<b>Krasnaya Dolina</b><br>May 2011"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.337448, 28.984614], 
    popup=dynamic_width_popup("<b>Ryabovo</b><br>June 2012"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [51.249206, 0.613427], 
    popup=dynamic_width_popup("<b>Leeds, Kent</b><br>November 2012"), 
    icon=folium.Icon(color=color_studies, icon=icon_studies, prefix="fa"), z_index_offset=z_index_studies
).add_to(m)

folium.Marker(
    [59.961646, 29.415822], 
    popup=dynamic_width_popup("<b>Lebyazhye</b><br>June 2015"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.972699, 29.332709], 
    popup=dynamic_width_popup("<b>Fort Krasnaya Gorka</b><br>June 2015"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.514869, 29.723454], 
    popup=dynamic_width_popup("<b>Korobitsyno</b><br>December 2016"), 
    icon=folium.Icon(color=color_studies, icon=icon_studies, prefix="fa"), z_index_offset=z_index_studies
).add_to(m)

folium.Marker(
    [60.125495, 31.073795], 
    popup=dynamic_width_popup("<b>Ladozhskoye Ozero</b><br>January 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.068705, 31.067436], 
    popup=dynamic_width_popup("<b>Kokkorevo</b><br>January 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.086581, 31.023509], 
    popup=dynamic_width_popup("<b>Vaganovo</b><br>January 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.316293, 29.907420], 
    popup=dynamic_width_popup("<b>Batovo</b><br>June 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.322303, 29.892540], 
    popup=dynamic_width_popup("<b>Daymishche</b><br>June 2017"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.811971, 30.573632], 
    popup=dynamic_width_popup("<b>Metallostroy</b><br>May 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.804076, 30.603509], 
    popup=dynamic_width_popup("<b>Ust-Izhora</b><br>November 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.755959, 31.051356], 
    popup=dynamic_width_popup("<b>Mga</b><br>August 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.756459, 30.972632], 
    popup=dynamic_width_popup("<b>Gory</b><br>August 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.810261, 30.895841], 
    popup=dynamic_width_popup("<b>Pavlovo</b><br>August 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [61.120024, 29.869007], 
    popup=dynamic_width_popup("<b>Kuznechnoye</b><br>June 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.94902, 34.10394], 
    popup=dynamic_width_popup("<b>Simferopol</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.19482, 33.35982], 
    popup=dynamic_width_popup("<b>Yevpatoria</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.61838, 33.52421], 
    popup=dynamic_width_popup("<b>Sevastopol</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.40088, 34.00105], 
    popup=dynamic_width_popup("<b>Simeiz</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.49079, 34.16347], 
    popup=dynamic_width_popup("<b>Yalta</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.67641, 34.41012], 
    popup=dynamic_width_popup("<b>Alushta</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.83925, 34.97677], 
    popup=dynamic_width_popup("<b>Sudak</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.03206, 35.38306], 
    popup=dynamic_width_popup("<b>Feodosia</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.35057, 36.46847], 
    popup=dynamic_width_popup("<b>Kerch</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.37495, 36.30082], 
    popup=dynamic_width_popup("<b>Bagerovo</b><br>September 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.164334, 30.540774], 
    popup=dynamic_width_popup("<b>Toksovo</b><br>August 2021"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [60.020719, 30.64572], 
    popup=dynamic_width_popup("<b>Vsevolozhsk</b><br>September 2021"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.738804, 29.750589], 
    popup=dynamic_width_popup("<b>Bol'shoye Zabrod'ye</b><br>January 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [42.878090, 74.612549], 
    popup=dynamic_width_popup("<b>Bishkek</b><br>March 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [40.186076, 44.515089], 
    popup=dynamic_width_popup("<b>Yerevan</b><br>April 2022"), 
    icon=folium.Icon(color=color_living, icon=icon_living, prefix="fa"), z_index_offset=z_index_living
).add_to(m)

folium.Marker(
    [39.878290, 44.576428], 
    popup=dynamic_width_popup("<b>Khor Virap</b><br>April 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [39.740772, 44.829733], 
    popup=dynamic_width_popup("<b>Yeraskh</b><br>April 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [39.684744, 45.233010], 
    popup=dynamic_width_popup("<b>Noravank</b><br>April 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [39.764008, 45.333238], 
    popup=dynamic_width_popup("<b>Yeghegnadzor</b><br>April 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [39.412256, 46.293627], 
    popup=dynamic_width_popup("<b>Halidzor</b><br>April 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [39.379510, 46.250143], 
    popup=dynamic_width_popup("<b>Tatev</b><br>April 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [39.501253, 46.433883], 
    popup=dynamic_width_popup("<b>Khndzoresk</b><br>April 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [39.690259, 45.468525], 
    popup=dynamic_width_popup("<b>Vayk</b><br>April 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [40.590901, 44.357533], 
    popup=dynamic_width_popup("<b>Aparan</b><br>May 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [41.005236, 44.634970], 
    popup=dynamic_width_popup("<b>Kopayr</b><br>May 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [41.050982, 44.616199], 
    popup=dynamic_width_popup("<b>Odzun</b><br>May 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [41.035175, 44.627274], 
    popup=dynamic_width_popup("<b>Horomayr</b><br>May 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [41.091856, 44.656903], 
    popup=dynamic_width_popup("<b>Alaverdi</b><br>May 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [41.093735, 44.711651], 
    popup=dynamic_width_popup("<b>Haghpat</b><br>May 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [40.547910, 44.958926], 
    popup=dynamic_width_popup("<b>Sevan</b><br>May 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [40.432781, 45.107905], 
    popup=dynamic_width_popup("<b>Hayravank</b><br>May 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.817031, 20.458710], 
    popup=dynamic_width_popup("<b>Belgrade</b><br>June 2022"), 
    icon=folium.Icon(color=color_living, icon=icon_living, prefix="fa"), z_index_offset=z_index_living
).add_to(m)

folium.Marker(
    [45.117230, 19.406807], 
    popup=dynamic_width_popup("<b>Erdevik</b><br>September 2022"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [1.281071, 103.847476], 
    popup=dynamic_width_popup("<b>Singapore</b><br>November 2022"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [59.355381, 17.945656], 
    popup=dynamic_width_popup("<b>Stockholm</b><br>November 2012"), 
    icon=folium.Icon(color=color_transit, icon=icon_transit, prefix="fa"), z_index_offset=z_index_transit
).add_to(m)

folium.Marker(
    [50.050418, 8.571717], 
    popup=dynamic_width_popup("<b>Fraknfurt am Main</b><br>November 2012"), 
    icon=folium.Icon(color=color_transit, icon=icon_transit, prefix="fa"), z_index_offset=z_index_transit
).add_to(m)

folium.Marker(
    [25.271117, 51.609183], 
    popup=dynamic_width_popup("<b>Doha</b><br>November 2016"), 
    icon=folium.Icon(color=color_transit, icon=icon_transit, prefix="fa"), z_index_offset=z_index_transit
).add_to(m)

folium.Marker(
    [25.249330, 55.352934], 
    popup=dynamic_width_popup("<b>Dubai</b><br>December 2016"), 
    icon=folium.Icon(color=color_transit, icon=icon_transit, prefix="fa"), z_index_offset=z_index_transit
).add_to(m)

folium.Marker(
    [53.891040, 27.550887], 
    popup=dynamic_width_popup("<b>Minsk</b><br>August 2018"), 
    icon=folium.Icon(color=color_transit, icon=icon_transit, prefix="fa"), z_index_offset=z_index_transit
).add_to(m)

folium.Marker(
    [34.870473, 33.608815], 
    popup=dynamic_width_popup("<b>Larnaca</b><br>December 2022"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [34.672267, 33.041601], 
    popup=dynamic_width_popup("<b>Limassol</b><br>December 2022"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [34.707135, 33.087090], 
    popup=dynamic_width_popup("<b>Germasogeia</b><br>December 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [34.664499, 32.887530], 
    popup=dynamic_width_popup("<b>Episkopi</b><br>December 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [34.603809, 32.952606], 
    popup=dynamic_width_popup("<b>Akrotiri</b><br>December 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [34.714060, 33.140423], 
    popup=dynamic_width_popup("<b>Agios Tychonas</b><br>December 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [34.755996, 33.093834], 
    popup=dynamic_width_popup("<b>Foinikaria</b><br>December 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [34.689343, 33.008743], 
    popup=dynamic_width_popup("<b>Kato Polemidia</b><br>December 2022"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.984613, 20.158889], 
    popup=dynamic_width_popup("<b>Stara Pazova</b><br>January 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [-37.817952, 144.968617], 
    popup=dynamic_width_popup("<b>Melbourne</b><br>May 2023"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [44.568427, 20.731114], 
    popup=dynamic_width_popup("<b>Umčari</b><br>June 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [49.600045, 6.133638], 
    popup=dynamic_width_popup("<b>Luxembourg</b><br>September 2023"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [48.860640, 2.337618], 
    popup=dynamic_width_popup("<b>Paris</b><br>September 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [48.728359, 2.274310], 
    popup=dynamic_width_popup("<b>Massy</b><br>September 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [48.720449, 2.248629], 
    popup=dynamic_width_popup("<b>Palaiseau</b><br>September 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [49.544829, 6.367586], 
    popup=dynamic_width_popup("<b>Remich</b><br>September 2023"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [49.515329, 6.304519], 
    popup=dynamic_width_popup("<b>Ellange</b><br>September 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [49.471395, 6.365505], 
    popup=dynamic_width_popup("<b>Schengen</b><br>September 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [49.474920, 6.387099], 
    popup=dynamic_width_popup("<b>Perl</b><br>September 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [49.461180, 6.378309], 
    popup=dynamic_width_popup("<b>Apach</b><br>September 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.255618, 19.845680], 
    popup=dynamic_width_popup("<b>Novi Sad</b><br>November 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [38.707531, -9.136466], 
    popup=dynamic_width_popup("<b>Lisbon</b><br>April 2024"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [38.781943, -9.497417], 
    popup=dynamic_width_popup("<b>Cabo da Roca</b><br>April 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.808512, 20.571532], 
    popup=dynamic_width_popup("<b>Slanci</b><br>July 2023"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.908259, 20.422440], 
    popup=dynamic_width_popup("<b>Kovilovo</b><br>March 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.875382, 20.450733], 
    popup=dynamic_width_popup("<b>Borča</b><br>March 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.810701, 20.597885], 
    popup=dynamic_width_popup("<b>Veliko Selo</b><br>April 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.762125, 20.623390], 
    popup=dynamic_width_popup("<b>Vinča</b><br>April 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.731436, 20.575265], 
    popup=dynamic_width_popup("<b>Leštane</b><br>April 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.754102, 20.260131], 
    popup=dynamic_width_popup("<b>Jakovo</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.790994, 20.270136], 
    popup=dynamic_width_popup("<b>Surčin</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.730052, 20.317110], 
    popup=dynamic_width_popup("<b>Ostružnica</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.678686, 20.304759], 
    popup=dynamic_width_popup("<b>Umka</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.660864, 20.310115], 
    popup=dynamic_width_popup("<b>Rucka</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.652332, 20.260206], 
    popup=dynamic_width_popup("<b>Barič</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.678620, 20.202372], 
    popup=dynamic_width_popup("<b>Zabrežje</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.67076, 20.16848],
    popup=dynamic_width_popup("<b>Urovci</b><br>May 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.722294, 20.228663], 
    popup=dynamic_width_popup("<b>Boljevci</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.664115, 20.353923], 
    popup=dynamic_width_popup("<b>Velika Moštanica</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.623750, 20.370699], 
    popup=dynamic_width_popup("<b>Meljak</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.601047, 20.400301], 
    popup=dynamic_width_popup("<b>Guncati</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.578149, 20.415967], 
    popup=dynamic_width_popup("<b>Barajevo</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.630595, 20.432750], 
    popup=dynamic_width_popup("<b>Stara Lipovica</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.682874, 20.436493], 
    popup=dynamic_width_popup("<b>Rušanj</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.694958, 20.489928], 
    popup=dynamic_width_popup("<b>Pinosava</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.705059, 20.511254], 
    popup=dynamic_width_popup("<b>Beli Potok</b><br>May 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.57693, 20.38727], 
    popup=dynamic_width_popup("<b>Baćevac</b><br>June 2024"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [44.87111, 20.64117], 
    popup=dynamic_width_popup("<b>Pančevo</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.80808, 20.70597], 
    popup=dynamic_width_popup("<b>Starčevo</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.75823, 20.73683], 
    popup=dynamic_width_popup("<b>Omoljica</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.81787, 20.87654], 
    popup=dynamic_width_popup("<b>Bavanište</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.74222, 20.97733], 
    popup=dynamic_width_popup("<b>Kovin</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.74222, 20.97733], 
    popup=dynamic_width_popup("<b>Kovin</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.66501, 20.92724], 
    popup=dynamic_width_popup("<b>Smederevo</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.66501, 20.92724], 
    popup=dynamic_width_popup("<b>Smederevo</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.65414, 20.825447], 
    popup=dynamic_width_popup("<b>Seone</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.64770, 20.75849], 
    popup=dynamic_width_popup("<b>Brestovik</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.67191, 20.72026], 
    popup=dynamic_width_popup("<b>Grocka</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.6953, 20.67443], 
    popup=dynamic_width_popup("<b>Zaklopača</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.73565, 20.60817], 
    popup=dynamic_width_popup("<b>Boleč</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.66478, 20.59215], 
    popup=dynamic_width_popup("<b>Vrčin</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.62510, 20.58102], 
    popup=dynamic_width_popup("<b>Ripanj</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.59294, 20.61369], 
    popup=dynamic_width_popup("<b>Mala Ivanča</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.56871, 20.64705], 
    popup=dynamic_width_popup("<b>Mali Požarevac</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.58495, 20.68899], 
    popup=dynamic_width_popup("<b>Drazanj</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.58346, 20.73556], 
    popup=dynamic_width_popup("<b>Umčari</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.61081, 20.72387], 
    popup=dynamic_width_popup("<b>Pudarci</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.63372, 20.69832], 
    popup=dynamic_width_popup("<b>Begaljica</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.94540, 20.21771], 
    popup=dynamic_width_popup("<b>Nova Pazova</b><br>June 2024"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.04787, 20.08061],
    popup=dynamic_width_popup("<b>Inđija</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.08112, 20.17692],
    popup=dynamic_width_popup("<b>Novi Karlovci</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.12733, 20.24097],
    popup=dynamic_width_popup("<b>Novi Slankamen</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.14137, 20.25791],
    popup=dynamic_width_popup("<b>Stari Slankamen</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.07113, 20.32517],
    popup=dynamic_width_popup("<b>Surduk</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.01921, 20.33316],
    popup=dynamic_width_popup("<b>Belegiš</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.98215, 20.28114],
    popup=dynamic_width_popup("<b>Stari Banovci</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.95076, 20.28270],
    popup=dynamic_width_popup("<b>Novi Banovci</b><br>June 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.90083, 20.87698],
    popup=dynamic_width_popup("<b>Dolovo</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.87947, 20.97341],
    popup=dynamic_width_popup("<b>Mramorak</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.83981, 21.04285],
    popup=dynamic_width_popup("<b>Deliblato</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.94112, 21.12513],
    popup=dynamic_width_popup("<b>Šušara</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.02173, 21.18425],
    popup=dynamic_width_popup("<b>Izbište</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.04306, 21.15672],
    popup=dynamic_width_popup("<b>Uljma</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.05270, 21.06952],
    popup=dynamic_width_popup("<b>Nikolinci</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.05107, 21.01602],
    popup=dynamic_width_popup("<b>Banatski Karlovac</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.03202, 20.86497],
    popup=dynamic_width_popup("<b>Vladimirovac</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.99002, 20.78345],
    popup=dynamic_width_popup("<b>Banatsko Novo Selo</b><br>August 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.87102, 20.20157],
    popup=dynamic_width_popup("<b>Ugrinovci</b><br>October 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.85519, 20.15667],
    popup=dynamic_width_popup("<b>Grmovac</b><br>October 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.82765, 20.22290],
    popup=dynamic_width_popup("<b>Dobanovci</b><br>October 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [34.68147, 32.60757], 
    popup=dynamic_width_popup("<b>Kouklia</b><br>November 2024"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [34.77424, 32.41946],
    popup=dynamic_width_popup("<b>Paphos</b><br>November 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [34.73532, 32.43397],
    popup=dynamic_width_popup("<b>Yeroskipou</b><br>November 2024"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [41.00852, 28.98004],
    popup=dynamic_width_popup("<b>Istanbul</b><br>January 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [40.94272, 27.31108],
    popup=dynamic_width_popup("<b>Nusratfakı</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [40.18096, 26.36060],
    popup=dynamic_width_popup("<b>Eceabat</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [40.15206, 26.40545],
    popup=dynamic_width_popup("<b>Çanakkale</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [39.95867, 26.25150],
    popup=dynamic_width_popup("<b>Tevfikiye</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [39.95724, 26.23947],
    popup=dynamic_width_popup("<b>Troy</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [41.24634, 29.03574],
    popup=dynamic_width_popup("<b>Kumköy</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [30.04441, 31.23572],
    popup=dynamic_width_popup("<b>Cairo</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [29.99838, 31.15944],
    popup=dynamic_width_popup("<b>Giza</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [29.97914, 31.13418],
    popup=dynamic_width_popup("<b>Giza necropolis</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [29.87117, 31.21674],
    popup=dynamic_width_popup("<b>Saqqarah necropolis</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [29.79029, 31.20931],
    popup=dynamic_width_popup("<b>Dahshur necropolis</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [25.69931, 32.63909],
    popup=dynamic_width_popup("<b>Luxor</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [31.20071, 29.89605],
    popup=dynamic_width_popup("<b>Alexandria</b><br>February 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.82961, 20.31408],
    popup=dynamic_width_popup("<b>Radiofar</b><br>April 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.88865, 20.53643],
    popup=dynamic_width_popup("<b>Ovča</b><br>May 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.23938, 19.88556],
    popup=dynamic_width_popup("<b>Petrovaradin</b><br>June 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.73999, 20.69607],
    popup=dynamic_width_popup("<b>Ivanovo</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.93559, 20.15159],
    popup=dynamic_width_popup("<b>Vojka</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.88942, 20.13837],
    popup=dynamic_width_popup("<b>Krnješevci</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.87330, 20.09689],
    popup=dynamic_width_popup("<b>Šimanovci</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.83590, 20.11464],
    popup=dynamic_width_popup("<b>Deč</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.79360, 20.15139],
    popup=dynamic_width_popup("<b>Petrovčić</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.78067, 20.20291],
    popup=dynamic_width_popup("<b>Bečmen</b><br>July 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.12867, 19.78729],
    popup=dynamic_width_popup("<b>Vrdnik</b><br>July 2025"),
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [44.65469, 20.20094],
    popup=dynamic_width_popup("<b>Obrenovac</b><br>August 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.65813, 20.20000],
    popup=dynamic_width_popup("<b>Rvati</b><br>August 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.67779, 20.53693],
    popup=dynamic_width_popup("<b>Šuplja Stena</b><br>October 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [44.69878, 20.55698],
    popup=dynamic_width_popup("<b>Zuce</b><br>October 2025"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [-24.64487, 25.90733],
    popup=dynamic_width_popup("<b>Gaborone</b><br>October 2025"),
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [-24.73433, 25.81268],
    popup=dynamic_width_popup("<b>Mokolodi</b><br>October 2025"),
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [43.85912, 18.42911],
    popup=dynamic_width_popup("<b>Sarajevo</b><br>March 2026"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [43.82255, 18.36909],
    popup=dynamic_width_popup("<b>Istočno Sarajevo</b><br>March 2026"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [43.82600, 18.33668],
    popup=dynamic_width_popup("<b>Ilidža</b><br>March 2026"),
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.24953, 19.75643], 
    popup=dynamic_width_popup("<b>Veternik</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.24153, 19.71337], 
    popup=dynamic_width_popup("<b>Futog</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.24116, 19.62194], 
    popup=dynamic_width_popup("<b>Begeč</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.27614, 19.56327], 
    popup=dynamic_width_popup("<b>Gložan</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.26758, 19.53162], 
    popup=dynamic_width_popup("<b>Čelarevo</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.25000, 19.38449], 
    popup=dynamic_width_popup("<b>Bačka Palanka</b><br>April 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [52.31553, 4.76024], 
    popup=dynamic_width_popup("<b>Schiphol</b><br>April 2026"), 
    icon=folium.Icon(color=color_transit, icon=icon_transit, prefix="fa"), z_index_offset=z_index_transit
).add_to(m)

folium.Marker(
    [51.67991, 39.20605], 
    popup=dynamic_width_popup("<b>Voronezh</b><br>September 2020"), 
    icon=folium.Icon(color=color_train, icon=icon_train, prefix="fa"), z_index_offset=z_index_train
).add_to(m)

folium.Marker(
    [47.21875, 39.68992], 
    popup=dynamic_width_popup("<b>Rostov-na-Donu</b><br>September 2020"), 
    icon=folium.Icon(color=color_train, icon=icon_train, prefix="fa"), z_index_offset=z_index_train
).add_to(m)

folium.Marker(
    [57.82383, 29.96130], 
    popup=dynamic_width_popup("<b>Dno</b><br>August 2018"), 
    icon=folium.Icon(color=color_train, icon=icon_train, prefix="fa"), z_index_offset=z_index_train
).add_to(m)

folium.Marker(
    [55.19486, 30.18578], 
    popup=dynamic_width_popup("<b>Vitebsk</b><br>August 2018"), 
    icon=folium.Icon(color=color_train, icon=icon_train, prefix="fa"), z_index_offset=z_index_train
).add_to(m)

folium.Marker(
    [54.67042, 25.28420], 
    popup=dynamic_width_popup("<b>Vilnius</b><br>August 2018"), 
    icon=folium.Icon(color=color_train, icon=icon_train, prefix="fa"), z_index_offset=z_index_train
).add_to(m)

folium.Marker(
    [-22.95169, -43.21054], 
    popup=dynamic_width_popup("<b>Rio de Janeiro</b><br>April 2026"), 
    icon=folium.Icon(color=color_work, icon=icon_work, prefix="fa"), z_index_offset=z_index_work
).add_to(m)

folium.Marker(
    [59.73217, 29.86828], 
    popup=dynamic_width_popup("<b>Ropsha</b><br>May 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.80168, 29.86488], 
    popup=dynamic_width_popup("<b>Uzigonty</b><br>May 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.82797, 29.87453], 
    popup=dynamic_width_popup("<b>Nizino</b><br>May 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.84253, 29.88744], 
    popup=dynamic_width_popup("<b>Sashino</b><br>May 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.85245, 29.89345], 
    popup=dynamic_width_popup("<b>Knyazevo</b><br>May 2018"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [59.90083, 30.51212], 
    popup=dynamic_width_popup("<b>Kudrovo</b><br>August 2020"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [60.04531, 30.44869], 
    popup=dynamic_width_popup("<b>Murino</b><br>March 2019"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel, prefix="fa"), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.19404, 19.89414], 
    popup=dynamic_width_popup("<b>Bukovac</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.13519, 19.90252], 
    popup=dynamic_width_popup("<b>Grgeteg</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.14630, 19.92400], 
    popup=dynamic_width_popup("<b>Velika Remeta</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

folium.Marker(
    [45.12076, 19.93840], 
    popup=dynamic_width_popup("<b>Krušedol Prnjavor</b><br>May 2026"), 
    icon=folium.Icon(color=color_travel, icon=icon_travel), z_index_offset=z_index_travel
).add_to(m)

In [11]:
m.save("travels.html")